<a href="https://colab.research.google.com/github/HassaanDeveloper/AI-Internship-Advance-Tasks/blob/main/Task4/AI_ML_Intern_Advanced_Task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q \
  langchain==0.3.25 \
  langchain-core==0.3.63 \
  langchain-community==0.3.24 \
  langchain-text-splitters==0.3.8 \
  faiss-cpu \
  sentence-transformers \
  transformers \
  accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 109.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.63 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-cor

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_core.documents import Document
from transformers import pipeline as hf_pipeline

print("✅ All imports successful!")

✅ All imports successful!


In [ ]:
corpus = [
    ("Machine Learning", """Machine learning is a branch of artificial intelligence that enables systems
to learn and improve from experience without being explicitly programmed.
It focuses on developing computer programs that can access data and use it to learn for themselves.
Machine learning algorithms include supervised learning, unsupervised learning, and reinforcement learning.
In supervised learning, models are trained on labeled data to make predictions on new unseen data.
Common supervised learning algorithms include linear regression, decision trees, and neural networks.
Unsupervised learning finds hidden patterns in data without pre-existing labels."""),

    ("Large Language Models", """Large language models (LLMs) are deep learning models trained on massive text datasets.
They can generate human-like text, answer questions, summarize documents, and write code.
Examples include GPT-4, Claude, Gemini, and LLaMA.
LLMs use the transformer architecture with attention mechanisms to understand context.
Fine-tuning allows LLMs to be specialized for specific tasks or domains.
Prompt engineering is the practice of crafting inputs to get desired outputs from LLMs."""),

    ("Retrieval Augmented Generation", """Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation.
Instead of relying solely on training data, RAG retrieves relevant documents at inference time.
The retrieved documents are passed as context to the language model along with the user query.
RAG reduces hallucinations by grounding responses in real retrieved documents.
The pipeline consists of a document store, an embedding model, a vector database, and an LLM.
RAG is widely used in chatbots, question answering systems, and enterprise search."""),

    ("Neural Networks", """Neural networks are computing systems inspired by biological neural networks in animal brains.
They consist of layers of interconnected nodes or neurons that process information.
Deep neural networks have many hidden layers enabling them to learn complex representations.
Convolutional Neural Networks (CNNs) are specialized for image recognition tasks.
Transformers replaced RNNs for NLP tasks due to their attention mechanism and parallelization.
Training neural networks uses backpropagation and gradient descent to minimize a loss function."""),

    ("Natural Language Processing", """Natural Language Processing (NLP) is a field of AI focused on human-computer language interaction.
It involves tasks like text classification, sentiment analysis, machine translation, and summarization.
Modern NLP uses deep learning models, especially transformers like BERT and GPT.
Tokenization is the process of breaking text into smaller units called tokens.
Named Entity Recognition (NER) identifies entities like names, places, and organizations in text.
NLP powers applications like chatbots, search engines, and voice assistants."""),
]

all_docs = [Document(page_content=text, metadata={"title": title})
            for title, text in corpus]

print(f"✅ Knowledge base ready! {len(all_docs)} documents loaded.")

✅ Knowledge base ready! 5 documents loaded.


In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(all_docs)

print(f"✅ Total chunks: {len(chunks)}")
print(f"\nSample chunk:\n{chunks[0].page_content}")

✅ Total chunks: 11

Sample chunk:
Machine learning is a branch of artificial intelligence that enables systems
to learn and improve from experience without being explicitly programmed.
It focuses on developing computer programs that can access data and use it to learn for themselves.


In [ ]:
print("Loading embedding model (downloads ~90MB once)...")

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

print(f"✅ Vector store ready! Total vectors: {vectorstore.index.ntotal}")

Loading embedding model (downloads ~90MB once)...


/tmp/ipykernel_7701/2862163716.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector store ready! Total vectors: 11


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Reload vectorstore (in case of restart)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)
vectorstore = FAISS.load_local(
    "faiss_index", embeddings,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("✅ RAG retriever ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ RAG retriever ready!


In [ ]:
def chat(question):
    docs = retriever.invoke(question)
    print(f"🧑 User: {question}")
    print(f"🤖 Bot (retrieved answer):")
    for i, doc in enumerate(docs):
        print(f"\n  [{i+1}] Source: {doc.metadata.get('title','?')}")
        print(f"       {doc.page_content[:250]}...")
    print("-" * 60)

print("=" * 60)
print("RAG CHATBOT — Retrieval Demonstration")
print("=" * 60 + "\n")

chat("What is machine learning?")
chat("How does it relate to neural networks?")
chat("What is RAG and why is it useful?")

RAG CHATBOT — Retrieval Demonstration

🧑 User: What is machine learning?
🤖 Bot (retrieved answer):

  [1] Source: Machine Learning
       Machine learning is a branch of artificial intelligence that enables systems
to learn and improve from experience without being explicitly programmed.
It focuses on developing computer programs that can access data and use it to learn for themselves....

  [2] Source: Machine Learning
       Machine learning algorithms include supervised learning, unsupervised learning, and reinforcement learning.
In supervised learning, models are trained on labeled data to make predictions on new unseen data....
------------------------------------------------------------
🧑 User: How does it relate to neural networks?
🤖 Bot (retrieved answer):

  [1] Source: Neural Networks
       Neural networks are computing systems inspired by biological neural networks in animal brains.
They consist of layers of interconnected nodes or neurons that process information.
Deep neu